# Lab 21 - LoRA / QLoRA Fine-tuning - **Llama-3.2-3B + Finance Alpaca**

**AICB-P2T3 - Ngày 21 - Chương 5 - Fine-tuning & An Toàn**

Mục tiêu: fine-tune **Llama-3.2-3B** với LoRA/QLoRA trên dataset **gbharti/finance-alpaca**, sau đó so sánh các rank khác nhau (`r=8`, `r=16`, `r=64`).

---

## Profile: `T4`

| Setting | Giá trị |
|---|---|
| Model | `unsloth/Llama-3.2-3B-Instruct-bnb-4bit` |
| Dataset | `gbharti/finance-alpaca` |
| Recommended GPU | T4 (16 GB) |
| Train batch / eval batch | 1 / 1 |
| Gradient accumulation | 8 (effective batch = 8) |
| Eval strategy | "no" |
| Max seq length cap | 1024 |
| Dataset samples | 200 by default (`DATASET_SAMPLE_SIZE`) |
| Estimated time | ~60 phút trên T4 với 200 samples |

> **T4-specific tweaks**: dùng model 3B đã quantize 4-bit, tắt eval-during-training, batch size = 1, manual eval fallback. Nếu muốn dùng toàn bộ ~68.9k rows của Finance Alpaca, đặt `DATASET_SAMPLE_SIZE = None` nhưng thời gian train sẽ lâu hơn rất nhiều.

## Lab Roadmap (khoảng 2 giờ)

| # | Bước | Output |
|---|------|--------|
| 1 | Dataset preparation (Finance Alpaca format, p95 tokenization, 90/10 split) | `train_ds`, `eval_ds` |
| 2 | Configure PEFT + load model 4-bit | model wrapped với LoRA |
| 3 | Train baseline `r=16` với SFTTrainer | adapter checkpoint |
| 4 | Rank experiment - train `r=8` và `r=64` | 2 adapter checkpoints |
| 5 | Evaluate (perplexity + qualitative finance prompts) | so sánh metrics |
| 6 | Save + viết report | deliverable |

## Deliverable

1. **3 LoRA adapter checkpoints** (`r=8, r=16, r=64`)
2. **Evaluation report** chứa training time, peak VRAM, eval perplexity, 5 qualitative before/after examples, training cost, kết luận về rank trade-off


## 0. Setup & Environment Check

In [ ]:
# Verify GPU is available before installing anything
!nvidia-smi

In [ ]:
import torch
assert torch.cuda.is_available(), "❌ GPU runtime cần được bật. Runtime > Change runtime type > GPU"
name = torch.cuda.get_device_name(0)
vram_gb = torch.cuda.get_device_properties(0).total_memory / 1e9
print(f"✓ GPU: {name}")
print(f"✓ VRAM: {vram_gb:.1f} GB")
print(f"✓ CUDA: {torch.version.cuda}")
print(f"✓ PyTorch: {torch.__version__}")

if vram_gb > 20:
    print("\n⚠️  Bạn có GPU lớn (>20GB VRAM). Nên dùng phiên bản BigGPU thay vì T4 này — sẽ nhanh hơn nhiều!")
elif 'T4' not in name:
    print(f"\n💡 Notebook này tối ưu cho T4 (16GB). GPU của bạn ({name}) có thể chạy được nhưng underutilized.")

In [ ]:
# Install Unsloth, TRL, datasets, evaluation libs
# trl >= 0.12 — accepts processing_class (matches transformers >= 4.46)
%%capture
!pip install -q --upgrade pip
!pip install -q "unsloth[colab-new] @ git+https://github.com/unslothai/unsloth.git"
!pip install -q --no-deps "trl>=0.12,<0.16" peft accelerate bitsandbytes
!pip install -q datasets matplotlib seaborn pandas

In [ ]:
# Optional: mount Google Drive to save checkpoints persistently
MOUNT_DRIVE = False  # Đổi thành True nếu muốn save vào Drive

if MOUNT_DRIVE:
    from google.colab import drive
    drive.mount('/content/drive')
    OUTPUT_DIR = '/content/drive/MyDrive/lab21_llama32_3b_finance_alpaca'
else:
    OUTPUT_DIR = '/content/lab21_llama32_3b_finance_alpaca'

import os
os.makedirs(OUTPUT_DIR, exist_ok=True)
print(f"OK Output dir: {OUTPUT_DIR}")


## 1. Dataset Preparation

Dataset dùng trong notebook này: **gbharti/finance-alpaca** từ HuggingFace.

Dataset có format Alpaca chuẩn với các cột `instruction`, `input`, `output` và `text`. Notebook sẽ dùng `instruction/input/output` để dựng lại prompt thống nhất, sau đó split 90/10 thành train/eval.


In [ ]:
# Load Finance Alpaca dataset từ HuggingFace
from datasets import load_dataset

DATASET_NAME = "gbharti/finance-alpaca"
DATASET_SAMPLE_SIZE = 200  # Đặt None nếu muốn dùng toàn bộ train split (~68.9k rows)

raw_full = load_dataset(DATASET_NAME, split="train")
raw_full = raw_full.filter(
    lambda ex: bool((ex.get("instruction") or "").strip()) and bool((ex.get("output") or "").strip())
)

raw = raw_full.shuffle(seed=42)
if DATASET_SAMPLE_SIZE is not None:
    raw = raw.select(range(min(DATASET_SAMPLE_SIZE, len(raw))))

print(f"OK Dataset: {DATASET_NAME}")
print(f"OK Loaded {len(raw)} samples from {len(raw_full)} valid rows")
print(f"OK Columns: {raw.column_names}")
print("\n--- Sample 0 ---")
print(raw[0])


In [ ]:
# Option B: dùng custom finance data (uncomment + chỉnh nếu cần)
# from datasets import Dataset
# my_data = [
#     {"instruction": "Explain compound interest.", "input": "", "output": "Compound interest is ..."},
#     # ... thêm examples tài chính của bạn
# ]
# raw = Dataset.from_list(my_data)
# print(f"OK Custom dataset: {len(raw)} samples")


In [ ]:
# Model + tokenizer config
from transformers import AutoTokenizer

MODEL_NAME = "unsloth/Llama-3.2-3B-Instruct-bnb-4bit"
MAX_SEQ_CAP = 1024  # hard cap for T4 profile

# Dùng all linear layers cho Llama để lấy stretch-goal bonus.
# Nếu giảng viên yêu cầu đúng lab spec q+v, đổi thành ["q_proj", "v_proj"].
LORA_TARGET_MODULES = [
    "q_proj", "k_proj", "v_proj", "o_proj",
    "gate_proj", "up_proj", "down_proj",
]
LORA_TARGET_MODULES_NOTE = "Target ALL layers for Llama (stretch goal); same modules for r=8/r=16/r=64."

tok = AutoTokenizer.from_pretrained(MODEL_NAME)
if tok.pad_token is None:
    tok.pad_token = tok.eos_token
EOS_TOKEN = tok.eos_token or ""

# Auto-detect column names - Finance Alpaca uses instruction/input/output
cols = raw.column_names
INSTRUCTION_COL = next((c for c in ["instruction", "prompt", "question"] if c in cols), None)
INPUT_COL       = next((c for c in ["input", "context"] if c in cols), None)
OUTPUT_COL      = next((c for c in ["output", "response", "answer"] if c in cols), None)
assert INSTRUCTION_COL and OUTPUT_COL, f"Không tìm thấy cột instruction/output trong: {cols}"
print(f"OK Cột dùng: instruction='{INSTRUCTION_COL}', input='{INPUT_COL}', output='{OUTPUT_COL}'")

ALPACA_TEMPLATE = """### Instruction:
{instruction}

### Input:
{input}

### Response:
{output}"""

ALPACA_TEMPLATE_NO_INPUT = """### Instruction:
{instruction}

### Response:
{output}"""

def format_alpaca(example):
    inp = example.get(INPUT_COL, "") if INPUT_COL else ""
    inp = inp or ""
    if inp.strip():
        text = ALPACA_TEMPLATE.format(
            instruction=example[INSTRUCTION_COL], input=inp,
            output=example[OUTPUT_COL])
    else:
        text = ALPACA_TEMPLATE_NO_INPUT.format(
            instruction=example[INSTRUCTION_COL],
            output=example[OUTPUT_COL])
    return {"text": text + EOS_TOKEN}

ds = raw.map(format_alpaca, remove_columns=raw.column_names)
print("\n--- Formatted sample ---")
print(ds[0]["text"][:500])


In [ ]:
# Token length analysis -> set max_seq_length = p95
import numpy as np
import matplotlib.pyplot as plt

lengths = [len(tok.encode(x["text"])) for x in ds]

p50 = int(np.percentile(lengths, 50))
p95 = int(np.percentile(lengths, 95))
p99 = int(np.percentile(lengths, 99))

token_stats = {
    "min": int(min(lengths)),
    "max": int(max(lengths)),
    "p50": p50,
    "p95": p95,
    "p99": p99,
}

print(f"Token length distribution:")
print(f"  min={token_stats['min']}, max={token_stats['max']}")
print(f"  p50={p50}, p95={p95}, p99={p99}")

# Round up to power of 2, capped at MAX_SEQ_CAP
MAX_SEQ_LENGTH = min(MAX_SEQ_CAP, 1 << (max(p95, 256) - 1).bit_length())
token_stats["chosen_max_seq_length"] = int(MAX_SEQ_LENGTH)
token_stats["max_seq_cap"] = int(MAX_SEQ_CAP)
print(f"\nOK Chọn max_seq_length = {MAX_SEQ_LENGTH} (cap = {MAX_SEQ_CAP})")

plt.figure(figsize=(8, 3))
plt.hist(lengths, bins=40, color='#0E2A52', edgecolor='white')
plt.axvline(p95, color='#C8102E', linestyle='--', label=f'p95 = {p95}')
plt.axvline(MAX_SEQ_LENGTH, color='green', linestyle='--', label=f'chosen = {MAX_SEQ_LENGTH}')
plt.xlabel('Tokens'); plt.ylabel('Count'); plt.title('Token length distribution')
plt.legend(); plt.tight_layout(); plt.show()


In [ ]:
# 90/10 train/eval split
split = ds.train_test_split(test_size=0.1, seed=42)
train_ds = split["train"]
eval_ds = split["test"]

dataset_info = {
    "dataset_name": DATASET_NAME,
    "sample_size_setting": DATASET_SAMPLE_SIZE,
    "valid_rows_before_sampling": int(len(raw_full)),
    "used_samples": int(len(raw)),
    "train_samples": int(len(train_ds)),
    "eval_samples": int(len(eval_ds)),
    "instruction_col": INSTRUCTION_COL,
    "input_col": INPUT_COL,
    "output_col": OUTPUT_COL,
    "split_seed": 42,
}

print(f"OK Train: {len(train_ds)}  |  Eval: {len(eval_ds)}")
print(dataset_info)


## 2. Load Model + Configure LoRA (Baseline `r=16`)

Dùng Unsloth `FastLanguageModel` để load **Llama-3.2-3B-Instruct** đã pre-quantize 4-bit (NF4). Unsloth tự động bật custom CUDA kernels.

PEFT config:
- `r=16` (rank - baseline)
- `lora_alpha=32` (scaling - alpha/r = 2)
- `target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]` theo recipe LoRA phổ biến cho Llama


In [ ]:
from unsloth import FastLanguageModel

def load_base_model():
    """Reload base model - gọi mỗi lần train với rank khác để start từ scratch."""
    model, tokenizer = FastLanguageModel.from_pretrained(
        model_name=MODEL_NAME,
        max_seq_length=MAX_SEQ_LENGTH,
        dtype=None,            # auto: bf16 trên Ampere+, fp16 trên T4
        load_in_4bit=True,     # QLoRA
    )
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    return model, tokenizer

def wrap_with_lora(model, r, alpha):
    """Wrap model với LoRA adapter."""
    return FastLanguageModel.get_peft_model(
        model,
        r=r,
        lora_alpha=alpha,
        target_modules=LORA_TARGET_MODULES,
        lora_dropout=0,
        bias="none",
        use_gradient_checkpointing="unsloth",  # -60% VRAM
        random_state=42,
    )

# Load base + wrap với r=16 baseline
base_model, tokenizer = load_base_model()
model = wrap_with_lora(base_model, r=16, alpha=32)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"\nOK Trainable: {trainable:,} ({100*trainable/total:.3f}% of {total:,})")


## 3. Train Baseline (`r=16`) với TRL SFTTrainer

- 3 epochs, cosine LR schedule, packing=True
- Train batch = 1, grad_accum = 8 (effective batch = 8)
- Eval strategy: "no"  # T4 không đủ VRAM cho mid-train eval


In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments, Trainer
import inspect, time, trl, transformers

LIBRARY_VERSIONS = {
    "trl": trl.__version__,
    "transformers": transformers.__version__,
    "torch": torch.__version__,
}
print(f"trl: {trl.__version__}  |  transformers: {transformers.__version__}")

# Monkey-patch: alias tokenizer -> processing_class for old TRL + new transformers.
import unsloth.models._utils as _u_utils
_underlying_init = getattr(_u_utils, "_original_trainer_init", Trainer.__init__)
if not getattr(_underlying_init, "_aliased", False):
    def _aliased_trainer_init(self, *args, **kwargs):
        if "tokenizer" in kwargs and "processing_class" not in kwargs:
            kwargs["processing_class"] = kwargs.pop("tokenizer")
        return _underlying_init(self, *args, **kwargs)
    _aliased_trainer_init._aliased = True
    _u_utils._original_trainer_init = _aliased_trainer_init
    if "tokenizer" not in inspect.signature(Trainer.__init__).parameters:
        _orig_t = Trainer.__init__
        def _t_init(self, *args, **kwargs):
            if "tokenizer" in kwargs and "processing_class" not in kwargs:
                kwargs["processing_class"] = kwargs.pop("tokenizer")
            return _orig_t(self, *args, **kwargs)
        _t_init._aliased = True
        Trainer.__init__ = _t_init
    print("OK Trainer.__init__ patched")

try:
    from trl import SFTConfig
    _HAS_SFTCONFIG = True
except ImportError:
    _HAS_SFTCONFIG = False

_TA_PARAMS = inspect.signature(TrainingArguments.__init__).parameters
_EVAL_KEY = "eval_strategy" if "eval_strategy" in _TA_PARAMS else "evaluation_strategy"
_SFT_PARAMS = inspect.signature(SFTTrainer.__init__).parameters
_SUPPORTS_OLD_KWARGS = "dataset_text_field" in _SFT_PARAMS

TRAINING_CONFIG = dict(
    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    eval_accumulation_steps=4,
    prediction_loss_only=True,
    gradient_accumulation_steps=8,
    warmup_ratio=0.10,
    num_train_epochs=3,
    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    fp16=not torch.cuda.is_bf16_supported(),
    bf16=torch.cuda.is_bf16_supported(),
    logging_steps=5,
    eval_steps=25,
    save_strategy="epoch",
    optim="adamw_8bit",
    weight_decay=0.01,
    seed=42,
    report_to="none",
)
TRAINING_CONFIG[_EVAL_KEY] = "no"  # T4 không đủ VRAM cho mid-train eval

print("Training config for report:")
print(TRAINING_CONFIG)

def make_trainer(model, tokenizer, train_ds, eval_ds, output_subdir, **overrides):
    base_kwargs = dict(TRAINING_CONFIG)
    base_kwargs["output_dir"] = os.path.join(OUTPUT_DIR, output_subdir)
    base_kwargs.update(overrides)

    if _HAS_SFTCONFIG:
        sft_extra = dict(dataset_text_field="text", packing=False, max_seq_length=MAX_SEQ_LENGTH)
        sft_params = inspect.signature(SFTConfig.__init__).parameters
        sft_extra = {k: v for k, v in sft_extra.items() if k in sft_params}
        valid_base = {k: v for k, v in base_kwargs.items() if k in sft_params}
        args = SFTConfig(**valid_base, **sft_extra)
    else:
        args = TrainingArguments(**base_kwargs)

    trainer_kwargs = {
        "model": model, "train_dataset": train_ds, "eval_dataset": eval_ds, "args": args,
    }
    if "processing_class" in _SFT_PARAMS:
        trainer_kwargs["processing_class"] = tokenizer
    else:
        trainer_kwargs["tokenizer"] = tokenizer
    if _SUPPORTS_OLD_KWARGS:
        trainer_kwargs.update(dict(dataset_text_field="text", max_seq_length=MAX_SEQ_LENGTH, packing=False))
    return SFTTrainer(**trainer_kwargs)

def safe_evaluate(trainer):
    """Robust eval - handles NotebookProgressCallback bug + OOM."""
    import gc as _gc
    _gc.collect(); torch.cuda.empty_cache()

    try:
        from transformers.utils.notebook import NotebookProgressCallback
        trainer.remove_callback(NotebookProgressCallback)
    except Exception:
        pass

    try:
        return trainer.evaluate()["eval_loss"]
    except (torch.cuda.OutOfMemoryError, RuntimeError) as e:
        print(f"Warning: trainer.evaluate() failed ({type(e).__name__}). Falling back to manual eval...")

    _gc.collect(); torch.cuda.empty_cache()
    m = trainer.model
    m.eval()
    eval_dl = trainer.get_eval_dataloader()
    total, n = 0.0, 0
    with torch.no_grad():
        for batch in eval_dl:
            batch = {k: v.to(m.device) for k, v in batch.items() if hasattr(v, "to")}
            out = m(**batch)
            total += out.loss.item(); n += 1
            del out; torch.cuda.empty_cache()
    return total / max(n, 1)


In [ ]:
# Train baseline r=16
import gc
torch.cuda.reset_peak_memory_stats()
trainer_16 = make_trainer(model, tokenizer, train_ds, eval_ds, "r16")

t0 = time.time()
result_16 = trainer_16.train()
wall_16 = time.time() - t0
vram_16 = torch.cuda.max_memory_allocated() / 1e9

print(f"\n✓ r=16 done in {wall_16/60:.1f} min, peak VRAM = {vram_16:.1f} GB")
trainer_16.save_model(os.path.join(OUTPUT_DIR, "r16"))

# Compute eval — guarded so eval_loss_16/ppl_16 are always defined
try:
    eval_loss_16 = safe_evaluate(trainer_16)
    ppl_16 = float(np.exp(eval_loss_16))
    print(f"✓ r=16 eval loss = {eval_loss_16:.4f}, perplexity = {ppl_16:.2f}")
except Exception as e:
    print(f"⚠ Eval failed: {e}. Setting NaN — recover later by reloading adapter.")
    eval_loss_16 = float("nan")
    ppl_16 = float("nan")

In [ ]:
# Plot training loss để detect overfitting, đồng thời lưu artifact cho REPORT.md
import pandas as pd

loss_curve_path = os.path.join(OUTPUT_DIR, "loss_curve.png")
training_log_path = os.path.join(OUTPUT_DIR, "training_log_r16.csv")

def plot_losses(log_history, title="Training Loss", save_path=None):
    df = pd.DataFrame(log_history)
    train = df[df["loss"].notna()] if "loss" in df else pd.DataFrame()
    eval_ = df[df["eval_loss"].notna()] if "eval_loss" in df else pd.DataFrame()
    plt.figure(figsize=(8, 4))
    if not train.empty:
        plt.plot(train["step"], train["loss"], label="train", color="#0E2A52")
    if not eval_.empty:
        plt.plot(eval_["step"], eval_["eval_loss"], label="eval", color="#C8102E", marker="o")
    plt.xlabel("Step"); plt.ylabel("Loss"); plt.title(title)
    plt.legend(); plt.grid(alpha=0.3); plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=160, bbox_inches="tight")
    plt.show()
    return df

training_log_df = plot_losses(trainer_16.state.log_history, title="Loss Curve - r=16", save_path=loss_curve_path)
training_log_df.to_csv(training_log_path, index=False)
print(f"OK Saved loss curve: {loss_curve_path}")
print(f"OK Saved training log: {training_log_path}")
print("T4 mode: eval-during-training tắt để tiết kiệm VRAM. Report cần ghi chú rằng loss curve chủ yếu là train loss.")


## 4. Rank Experiment - `r=8` vs `r=64`

Train 2 adapters thêm với rank khác để hiểu trade-off:
- **`r=8`** - ít trainable params hơn, train nhanh hơn, thường ít VRAM hơn
- **`r=64`** - nhiều trainable params hơn, có thể học domain tốt hơn nhưng tốn thời gian/VRAM hơn


In [ ]:
def train_one_rank(r, alpha):
    """Train fresh adapter với rank cụ thể, return metrics."""
    gc.collect(); torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    base_m, tok = load_base_model()
    m = wrap_with_lora(base_m, r=r, alpha=alpha)
    trainable = sum(p.numel() for p in m.parameters() if p.requires_grad)

    tr = make_trainer(m, tok, train_ds, eval_ds, f"r{r}")
    t0 = time.time()
    tr.train()
    wall = time.time() - t0
    vram = torch.cuda.max_memory_allocated() / 1e9

    # Save adapter BEFORE eval — eval may OOM but we want the checkpoint
    tr.save_model(os.path.join(OUTPUT_DIR, f"r{r}"))
    print(f"✓ r={r} adapter saved. Computing eval loss...")

    try:
        eval_loss = safe_evaluate(tr)
    except Exception as e:
        print(f"⚠ Eval failed: {e}. Setting eval_loss=NaN.")
        eval_loss = float('nan')

    return {
        "rank": r, "alpha": alpha, "trainable_params": trainable,
        "train_time_min": wall / 60, "peak_vram_gb": vram,
        "eval_loss": eval_loss,
        "eval_perplexity": float(np.exp(eval_loss)) if not np.isnan(eval_loss) else float('nan'),
        "trainer": tr, "model": m, "tokenizer": tok,
    }

In [ ]:
# Cleanup baseline before training rank experiments
del trainer_16, model, base_model
gc.collect(); torch.cuda.empty_cache()

print("\n========== Training r=8 ==========")
exp_8 = train_one_rank(r=8, alpha=16)

In [ ]:
# Cleanup r=8 references
del exp_8["trainer"], exp_8["model"]
gc.collect(); torch.cuda.empty_cache()

print("\n========== Training r=64 ==========")
exp_64 = train_one_rank(r=64, alpha=128)

In [ ]:
# Cleanup r=64 references
del exp_64["trainer"], exp_64["model"]
gc.collect(); torch.cuda.empty_cache()

# Rubric yêu cầu thêm base-model perplexity để so với các LoRA ranks.
print("\n========== Evaluating base model ==========")
base_eval_loss = float("nan")
base_eval_ppl = float("nan")
try:
    base_eval_model, base_eval_tok = load_base_model()
    base_eval_trainer = make_trainer(base_eval_model, base_eval_tok, train_ds, eval_ds, "base_eval")
    base_eval_loss = safe_evaluate(base_eval_trainer)
    base_eval_ppl = float(np.exp(base_eval_loss))
    print(f"OK Base eval loss = {base_eval_loss:.4f}, perplexity = {base_eval_ppl:.2f}")
except Exception as e:
    print(f"Warning: base eval failed: {e}. REPORT.md cần ghi rõ lý do nếu giá trị là NaN.")
finally:
    for var_name in ["base_eval_trainer", "base_eval_model", "base_eval_tok"]:
        if var_name in globals():
            del globals()[var_name]
    gc.collect(); torch.cuda.empty_cache()

# Build summary table for REPORT.md
results = [
    {"rank": "Base", "rank_order": 0, "alpha": np.nan, "trainable_params": 0,
     "train_time_min": 0.0, "peak_vram_gb": np.nan,
     "eval_loss": base_eval_loss, "eval_perplexity": base_eval_ppl},
    {"rank": 8, "rank_order": 8, **{k: v for k, v in exp_8.items() if k not in ("trainer", "model", "tokenizer")}},
    {"rank": 16, "rank_order": 16, "alpha": 32, "trainable_params": int(trainable),
     "train_time_min": wall_16/60, "peak_vram_gb": vram_16,
     "eval_loss": eval_loss_16, "eval_perplexity": ppl_16},
    {"rank": 64, "rank_order": 64, **{k: v for k, v in exp_64.items() if k not in ("trainer", "model", "tokenizer")}},
]
summary_df = pd.DataFrame(results).sort_values("rank_order").reset_index(drop=True)
summary_display_df = summary_df.drop(columns=["rank_order"])
print("\n=== Rank Experiment Summary ===")
print(summary_display_df.to_string(index=False))


## 5. Evaluation - Qualitative Comparison

Generate finance test prompts và so sánh base model với adapter fine-tuned `r=16` trên 5 prompts (chạy full 10 nếu thời gian cho phép).


In [ ]:
TEST_PROMPTS = [
    "Explain the difference between APR and APY in simple terms.",
    "When should someone prioritize paying off high-interest debt over investing?",
    "What are the main risks of buying options close to expiration?",
    "How does diversification reduce portfolio risk?",
    "Explain capital gains tax at a high level for a new investor.",
    "What should a beginner consider before buying penny stocks?",
    "Compare an ETF and a mutual fund for long-term investing.",
    "Why is an emergency fund important before investing?",
    "Why do bond prices usually fall when interest rates rise?",
    "What questions should someone ask before refinancing a mortgage?",
]
print(f"OK {len(TEST_PROMPTS)} test prompts")


In [ ]:
from peft import PeftModel

def generate_response(model, tokenizer, prompt, max_new_tokens=256):
    FastLanguageModel.for_inference(model)
    text = ALPACA_TEMPLATE_NO_INPUT.format(instruction=prompt, output="")
    inputs = tokenizer(text, return_tensors="pt").to("cuda")
    out = model.generate(**inputs, max_new_tokens=max_new_tokens,
        temperature=0.7, top_p=0.9, do_sample=True,
        pad_token_id=tokenizer.eos_token_id)
    full = tokenizer.decode(out[0], skip_special_tokens=True)
    return full.split("### Response:")[-1].strip()

# Reload base + r=16 adapter
base_for_eval, tok_for_eval = load_base_model()
ft_model = PeftModel.from_pretrained(base_for_eval, os.path.join(OUTPUT_DIR, "r16"))

qualitative_results = []
for i, prompt in enumerate(TEST_PROMPTS[:5]):
    print(f"\n=== Prompt {i+1}: {prompt[:80]}...")
    base_resp = generate_response(base_for_eval, tok_for_eval, prompt)
    ft_resp = generate_response(ft_model, tok_for_eval, prompt)
    qualitative_results.append({
        "example_id": i + 1,
        "prompt": prompt,
        "base_response": base_resp,
        "finetuned_r16_response": ft_resp,
        "judgment": "TODO: improved / same / degraded",
        "notes": "TODO: nhận xét ngắn để đưa vào REPORT.md",
    })
    print(f"  BASE: {base_resp[:250]}...")
    print(f"  FT  : {ft_resp[:250]}...")


In [ ]:
qual_df = pd.DataFrame(qualitative_results)
qual_path = os.path.join(OUTPUT_DIR, "qualitative_comparison.csv")
qual_df.to_csv(qual_path, index=False)
print(f"OK Saved qualitative comparison: {qual_path}")
print(qual_df[["example_id", "prompt", "judgment", "notes"]].head())
